In [2]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
from pypdf import PdfReader
from dotenv import load_dotenv

# Load OPENAI_API_KEY from a local .env file (see .env.example) - never hard-code a key
load_dotenv()

client = OpenAI()

class PaperData(BaseModel):
    title: str = Field(description="The title of the paper")
    abstract_summary: List[str] = Field(description="One sentence summary of the abstract")
    intro_backgrounds_summary: str = Field(description="One sentence summary of the intro and backgrounds sections")
    methods_summary: str = Field(description="One sentence summary of the methods section")
    results_summary: str = Field(description="One sentence summary of the results section")
    discussion_summary: str = Field(description="One sentence summary of the discussion section")
    

def load_pdf_text(file_path):
    '''Loads text from a PDF file.'''
    # creating a pdf reader object
    reader = PdfReader(file_path)

    # extracting text from page
    text = "\n\n".join([page.extract_text() for page in reader.pages])
    
    return text    

In [3]:
import os, urllib.request

def download_pdf(url, dest):
    """Download a PDF to `dest` if it is not already there (cross-platform; no wget)."""
    if os.path.exists(dest):
        print(f"Already present: {dest}")
        return
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
        f.write(r.read())
    print(f"Downloaded: {dest}")

download_pdf("https://arxiv.org/pdf/2510.26493", "../assets/paper.pdf")

Already present: ../assets/paper.pdf


In [4]:
paper_raw_text = load_pdf_text("../assets/paper.pdf")

response = client.responses.parse(
    model="gpt-5.6-luna",
    input=f"extract the information from the paper: {paper_raw_text}",
    text_format=PaperData
)

print(response)

ParsedResponse(id='resp_0152f5274b5ec8f2006ab020c2a27887d2b497b4c3435959c4', created_at=1789927618.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-luna', object='response', output=[ResponseReasoningItem(id='rs_0152f5274b5ec8f2006ab020c389f087d2b788afc751eea1e2', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqsCDI6znKB72EDrN4Vk5C3RGRznQnxfQHSqtRsI4E911r9NgpypaSHK3ZFFDj7RiyG2CnFJ4QAsaAOY3RzE_xTB1bxTVVR0-hgeem-Hex3-4c70UC-qjojEZ2cw5YlLxYk3NJWOfObXeI1IwMyoGM7cMraThGPHzRYCDNEWFLU8DT7s7v5Iqz4ZVTyGuAvnyH0AoQi4FLNY24fVdZcvaBXFIpF0mitpss7sGL-_nIiOfIt1J_8BowYb2oMXzWhGe8Nv6pKn4ISgRZGnnvcCPYSr_k2XphYJisx95rC9ipyxwgLTnNEBwj9Zx-8o36jMDJ9P54WjsqV5PIZF2aTP_4qBMiB7u43TSuXiaX0yd71_bv3Jjd64cHylhMDRvkISq6_2Qw_r-aNlxuoySGlyEUvSWcbgyGJnypK-FPXWJP1dRobyg0U1BMrihqLuZJ2zFb-9IsdEz1hv7YKdODYqwWghNRqs6-4CIcXST5zzTgQjHuULZwmqIYgXvyDutTVUmbRsiMPJsDgotjuyQ4Fad_r1xwhLZ0bUN_T3eWgaboPkrEE2JrpRgm5wkUoJlhCX9_7HEkr18Zyy2JE-dhWj9kqMauxbskpoG6LCrCFckHwwjttqGpc19Rx

In [5]:
response.output_parsed

PaperData(title='Context Engineering 2.0: The Context of Context Engineering', abstract_summary=['The paper frames context engineering as a long-evolving discipline for reducing information entropy and bridging human intent with machine understanding, traces its development across four intelligence-driven eras, and proposes design principles for collecting, managing, and using context in AI systems.'], intro_backgrounds_summary='The introduction situates modern context engineering within a broader history spanning human-computer interaction, ubiquitous computing, and context-aware systems, arguing that advances in machine intelligence progressively reduce the human effort required to translate intentions into machine-understandable contexts.', methods_summary='The authors develop a conceptual and theoretical framework by formalizing entities, interactions, context, and context engineering; comparing historical eras; and synthesizing design strategies for context collection and storage,

In [6]:
from IPython.display import Markdown, display

md = f"""
# {response.output_parsed.title}

## Abstract Summary
{'; '.join(response.output_parsed.abstract_summary)}

## Introduction & Backgrounds Summary
{response.output_parsed.intro_backgrounds_summary}

## Methods Summary
{response.output_parsed.methods_summary}

## Results Summary
{response.output_parsed.results_summary}

## Discussion Summary
{response.output_parsed.discussion_summary}
"""

display(Markdown(md))


# Context Engineering 2.0: The Context of Context Engineering

## Abstract Summary
The paper frames context engineering as a long-evolving discipline for reducing information entropy and bridging human intent with machine understanding, traces its development across four intelligence-driven eras, and proposes design principles for collecting, managing, and using context in AI systems.

## Introduction & Backgrounds Summary
The introduction situates modern context engineering within a broader history spanning human-computer interaction, ubiquitous computing, and context-aware systems, arguing that advances in machine intelligence progressively reduce the human effort required to translate intentions into machine-understandable contexts.

## Methods Summary
The authors develop a conceptual and theoretical framework by formalizing entities, interactions, context, and context engineering; comparing historical eras; and synthesizing design strategies for context collection and storage, management, abstraction, sharing, selection, and usage across AI systems.

## Results Summary
The paper proposes four stages of context engineering—primitive computation, agent-centric intelligence, human-level intelligence, and speculative superhuman intelligence—and identifies practical approaches including multimodal sensing, layered memory, context isolation, self-baking, cross-agent sharing, adaptive context selection, proactive need inference, and lifelong context management.

## Discussion Summary
The discussion highlights unresolved challenges in scalable storage, long-context processing, semantic consistency, system stability, evaluation, privacy, and proactive context collection, and calls for a semantic operating system that actively manages, updates, explains, and evolves context over time.


In [7]:
import pandas as pd
import os

def upsert_paper_to_csv(paper_data, csv_path="papers_database.csv", key_column="title"):
    """
    Inserts or updates a paper's data in a CSV file database. If the file does not exist,
    it creates it. If the file exists, it updates the entry if the key exists, or appends 
    the new paper if not.

    Parameters:
        paper_data: an object with attributes corresponding to extracted fields
        csv_path: output file path for CSV
        key_column: column name to use as unique identifier (default: "title")
    """
    # Prepare data dictionary, flattening abstract_summary if it's a list
    paper_dict = {
        "title": getattr(paper_data, "title", ""),
        "abstract_summary": "; ".join(paper_data.abstract_summary) if isinstance(getattr(paper_data, "abstract_summary", ""), list) else getattr(paper_data, "abstract_summary", ""),
        "intro_backgrounds_summary": getattr(paper_data, "intro_backgrounds_summary", ""),
        "methods_summary": getattr(paper_data, "methods_summary", ""),
        "results_summary": getattr(paper_data, "results_summary", ""),
        "discussion_summary": getattr(paper_data, "discussion_summary", ""),
    }

    # If CSV exists, load and update if key matches; otherwise create new
    if os.path.isfile(csv_path):
        df = pd.read_csv(csv_path)
        # Check if key (e.g., title) already in db
        if paper_dict[key_column] in df[key_column].values:
            # Update existing entry
            df.loc[df[key_column] == paper_dict[key_column], list(paper_dict.keys())] = list(paper_dict.values())
            updated = True
        else:
            # Append if not found
            df = pd.concat([df, pd.DataFrame([paper_dict])], ignore_index=True)
            updated = False
    else:
        # Create new dataframe
        df = pd.DataFrame([paper_dict])
        updated = False

    df.to_csv(csv_path, index=False)
    if updated:
        print(f"Updated entry in '{csv_path}' for {key_column}='{paper_dict[key_column]}'")
    else:
        print(f"Added new entry to '{csv_path}' for {key_column}='{paper_dict[key_column]}'")

# Example usage:
upsert_paper_to_csv(response.output_parsed)

Updated entry in 'papers_database.csv' for title='Context Engineering 2.0: The Context of Context Engineering'


In [8]:
import pandas as pd

df = pd.read_csv("../assets/papers_database.csv")

df

,title,abstract_summary,intro_backgrounds_summary,methods_summary,results_summary,discussion_summary
0,Context Engineering 2.0: The Context of Contex...,The paper defines and situates context enginee...,The introduction motivates context engineering...,The paper formalizes context with mathematical...,"Rather than empirical experiments, the paper p...",The discussion highlights key challenges for l...
1,Efﬁcient Estimation of Word Representations in...,The paper introduces two efficient model archi...,The authors motivate moving beyond atomic word...,They propose two log-linear architectures—CBOW...,"Experimentally, Skip-gram yields the best sema...","The paper concludes that very simple, computat..."


In [9]:
download_pdf("https://arxiv.org/pdf/1301.3781", "../assets/paper3.pdf")

Already present: ../assets/paper3.pdf


In [10]:
def extract_paper_data_workflow(paper_path: str):
    paper_raw_text = load_pdf_text(paper_path)

    response = client.responses.parse(
        model="gpt-5.6-luna",
        input=f"extract the information from the paper: {paper_raw_text}",
        text_format=PaperData
    )
    
    output_parsed = response.output_parsed

    print(output_parsed)
    
    upsert_paper_to_csv(output_parsed)
    
    return output_parsed

extract_paper_data_workflow("../assets/paper3.pdf")

title='Efficient Estimation of Word Representations in Vector Space' abstract_summary=['The paper introduces the CBOW and Skip-gram architectures for efficiently learning high-quality continuous word vectors from very large corpora, demonstrating improved syntactic and semantic performance at substantially lower computational cost.'] intro_backgrounds_summary='The paper motivates distributed word representations as a way to capture semantic and syntactic relationships beyond atomic word identities, reviews neural language-model approaches, and aims to learn accurate vectors from billions of words using scalable architectures that preserve linear word relationships.' methods_summary='The authors compare feedforward and recurrent neural language models with two new log-linear architectures—Continuous Bag-of-Words, which predicts a target word from averaged context vectors, and Skip-gram, which predicts surrounding words from a target word—trained with stochastic gradient methods, hierarc

PaperData(title='Efficient Estimation of Word Representations in Vector Space', abstract_summary=['The paper introduces the CBOW and Skip-gram architectures for efficiently learning high-quality continuous word vectors from very large corpora, demonstrating improved syntactic and semantic performance at substantially lower computational cost.'], intro_backgrounds_summary='The paper motivates distributed word representations as a way to capture semantic and syntactic relationships beyond atomic word identities, reviews neural language-model approaches, and aims to learn accurate vectors from billions of words using scalable architectures that preserve linear word relationships.', methods_summary='The authors compare feedforward and recurrent neural language models with two new log-linear architectures—Continuous Bag-of-Words, which predicts a target word from averaged context vectors, and Skip-gram, which predicts surrounding words from a target word—trained with stochastic gradient met

In [11]:
import pandas as pd


df = pd.read_csv("../assets/papers_database.csv")
df

,title,abstract_summary,intro_backgrounds_summary,methods_summary,results_summary,discussion_summary
0,Context Engineering 2.0: The Context of Contex...,The paper defines and situates context enginee...,The introduction motivates context engineering...,The paper formalizes context with mathematical...,"Rather than empirical experiments, the paper p...",The discussion highlights key challenges for l...
1,Efﬁcient Estimation of Word Representations in...,The paper introduces two efficient model archi...,The authors motivate moving beyond atomic word...,They propose two log-linear architectures—CBOW...,"Experimentally, Skip-gram yields the best sema...","The paper concludes that very simple, computat..."


In [12]:
tool_schema = {
    "type": "function",
    "name": "extract_paper_data_workflow",
    "description": "Extracts data from a paper",
    "parameters": {
        "type": "object",
        "properties": {
            "paper_path": {
                "type": "string",
                "description": "The path to the paper"
            }
        },
        "required": ["paper_path"],
    }
}

tool_schema

{'type': 'function',
 'name': 'extract_paper_data_workflow',
 'description': 'Extracts data from a paper',
 'parameters': {'type': 'object',
  'properties': {'paper_path': {'type': 'string',
    'description': 'The path to the paper'}},
  'required': ['paper_path']}}

In [13]:
download_pdf("https://arxiv.org/pdf/2412.14161", "./test_agent_paper.pdf")

Already present: ./test_agent_paper.pdf


In [14]:
response = client.responses.create(
    model="gpt-5.6-luna",
    instructions="You are an extraction agent, users will give you paths to files for pdfs\
                  and you will extract the data from the paper using the:\
                  extract_paper_data function.",
    tools=[tool_schema],
    input="Extract data from this paper: ./test_agent_paper.pdf"
)

response.output_text

''

In [15]:
response.output

[ResponseReasoningItem(id='rs_0f1e5cd0d02ee0a1006ab020ce765887d2a8382bfc52e854f0', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqsCDOuiz6kk9RNhODVdX9FukyujJGAxOdNEU9SlYl3b4tNKS3Gwxrt7XLqNOleO-7lTDzqFdFE-dlHnmfFwQU6F7NR_M5bQD84Qd1BT7mDSjVbYLW481B9uCWO3lyQGEKxlK1h72lhjKVfaBqFNQ5CnyIb39Z09-C4FxojWc4LRp1ySdtmcngFUCxReHmvCrpAZDli5_NeBEvfOAukwpV_L-Fa0GEv-MAeNMl9KrlCoaZB3gStXh-bf3utrOO21a0QE-kCKO9JE6mSOhga6sSqhqBgx5PlJymd_pJMEMnPDX5a9MeOkObuAU-NuyKAP5sYGEW9Ej3RTWifkQrN7hxKJzWAe6JfuYIHr-GVCsYNg8gWmWyAJBO6DxA069w2qT2gqrir-E34BEfLTIJDtSj1gfmcuEHs0pqfMAsjHZsAPLtZWbYwtPTCyT4GxsvWS0-8htu0EFqm5Wda7EpUR_cW1KmYWYYNeYsqST5dOw8kB_Ql87hNNSNpwsu9Cpxrbn__o27q6Ik2M5H8wnsH6DezDTjl0ZhXBVtCLv86S-JPtvJBkQ3eAeSmhqbtm5TmdqpU5F90JYgj-HFouFsyzZkpVE1U7JGS9AxmgNYGgbdHSsBkkEB-F2U01kP8YhirgQuKBAO6CLr_1M_tMNn6QFYJUbb2SZ1nOwwuUC3wgT9mm39PsoSqdj513xSJKKO8_P6_9bqMvOMamqyaqCHHIOFjLXoOtBO1ftLIWBVZsZab2wrVKLKl7tE4ISELNvEklFJpRpdct49618hmU6jqlPl3_d4oJbKmASgGxNaKqPuz0IfUfc9q8Wr71GwHC7-vVXKtNA20XWnTTmsr4A6